# 🧹 Module 2 — Nettoyage & Wrangling des Données
## *Data Cleaning & Wrangling*

---

**Projet / Project :** Social Media vs Traditional Media  
**Auteure / Author :** Inès Amdjahed — Master 2 Info-Com, Paris Nanterre  
**Date :** 2026-03-11  

---

### 🇫🇷 Objectif de ce notebook

Ce notebook constitue la **deuxième étape du pipeline de traitement des données**.
Après la collecte (Module 1), on prépare les données pour l'analyse exploratoire.

Le nettoyage des données (*data cleaning*) est une étape critique en data science :
des données mal nettoyées produisent des analyses erronées (*garbage in, garbage out*).

**Pipeline complet du projet :**
```
Module 1 — Collecte        → scrape_ourworldindata.py + collect_cnc_mediametrie.py
Module 2 — Cleaning        → ce notebook  ← ON EST ICI
Module 3 — EDA & Dataviz   → 02_eda_exploratory.ipynb
Module 4 — Dashboard       → Power BI
```

**Opérations réalisées dans ce notebook :**
1. Chargement de tous les fichiers raw collectés en Module 1
2. Renommage des colonnes vers des noms standardisés (snake_case)
3. Typage : vérification et correction des types de données
4. Filtrage : restriction à la période 2012-2024
5. Harmonisation : toutes les durées en minutes/jour
6. Feature engineering : nouvelles variables (total, indices base 100, écart générationnel)
7. Sauvegarde dans `02_data/processed/`

### 🇬🇧 Objective
Second step of the data pipeline. Load → rename → type → filter → harmonize → engineer → save.

---

### 📦 Datasets traités

| # | Source | Contenu | Hypothèse |
|---|---|---|---|
| 1 | GWI / DataReportal | Temps réseaux sociaux — tendance mondiale 2012-2024 | H1 |
| 2 | eMarketer / OWID | Temps médias numériques USA 2008-2018 | H1 |
| 3 | Rapports plateformes | MAU Facebook / Instagram / TikTok / YouTube... | H2, H3 |
| 4 | GWI Q3 2024 | Comparaison internationale par pays 2024 | H3 |
| 5 | CNC / Médiamétrie | TV France par tranche d'âge 2009-2024 | H1, H3 |
| 6 | FUSION | Dataset consolidé GWI × TV France (pour tester H1) | H1 |


---
## ⚙️ 0. Configuration & Imports

In [15]:
import pandas as pd
import numpy as np
import os
import glob
import warnings
warnings.filterwarnings('ignore')

# Chemins relatifs depuis 03_notebooks/
RAW_DIR       = os.path.join('..', '02_data', 'raw')
PROCESSED_DIR = os.path.join('..', '02_data', 'processed')
os.makedirs(PROCESSED_DIR, exist_ok=True)

YEAR_START = 2012
YEAR_END   = 2024

print('✅ Configuration chargée')
print(f'   RAW_DIR       : {os.path.abspath(RAW_DIR)}')
print(f'   PROCESSED_DIR : {os.path.abspath(PROCESSED_DIR)}')
print(f'   Période       : {YEAR_START} → {YEAR_END}')

✅ Configuration chargée
   RAW_DIR       : /home/inesajd/social-media-vs-traditional-media/02_data/raw
   PROCESSED_DIR : /home/inesajd/social-media-vs-traditional-media/02_data/processed
   Période       : 2012 → 2024


---
## 🔧 1. Fonctions utilitaires

**FR :** `load_latest_raw()` charge le fichier raw le plus récent pour un préfixe donné.
`save_processed()` sauvegarde un DataFrame nettoyé dans `processed/`.

**EN :** Helper functions: load latest raw CSV, save cleaned DataFrame.

In [16]:
def load_latest_raw(prefix: str) -> pd.DataFrame:
    """
    FR : Charge le fichier CSV le plus récent correspondant au préfixe.
         Les fichiers raw ont un horodatage dans leur nom (ex: _2026-03-11.csv).
         sorted()[-1] = dernier par ordre alphabétique = le plus récent
         (les dates ISO 8601 sont triables alphabétiquement).
    EN : Loads most recent CSV matching the prefix.
    """
    pattern = os.path.join(RAW_DIR, f'{prefix}*.csv')
    files   = sorted(glob.glob(pattern))
    if not files:
        raise FileNotFoundError(
            f"❌ Fichier introuvable : '{prefix}'\n"
            f"   → As-tu lancé les scripts de collecte (Module 1) ?"
        )
    latest = files[-1]
    df = pd.read_csv(latest, encoding='utf-8')
    print(f'   📂 {os.path.basename(latest)}  →  {df.shape}')
    return df


def save_processed(df: pd.DataFrame, name: str) -> str:
    """
    FR : Sauvegarde dans 02_data/processed/. Pas d'horodatage — fichiers stables.
    EN : Saves to processed/. No timestamp — stable versioned files.
    """
    filepath = os.path.join(PROCESSED_DIR, f'{name}.csv')
    df.to_csv(filepath, index=False, encoding='utf-8')
    print(f'   💾 {name}.csv  →  {df.shape}')
    return filepath


print('✅ Fonctions prêtes')

✅ Fonctions prêtes


---
## 📊 2. Dataset 1 — Temps réseaux sociaux (GWI mondial 2012-2024)

**FR :** Variable indépendante principale (H1). Source : GWI panel 53 pays, internautes 16-64 ans.

**EN :** Main independent variable (H1). GWI panel, 53 countries.

In [17]:
print('=' * 55)
print('DATASET 1 — Temps réseaux sociaux (GWI mondial)')
print('=' * 55)

df_social_raw = load_latest_raw('gwi_gwi_social_media_global_trend')
df_social = df_social_raw.copy()

# Renommage snake_case
df_social = df_social.rename(columns={
    'entity':               'country',
    'social_media_minutes': 'social_media_min_day',
    'social_media_hours':   'social_media_hrs_day',
    'population':           'population_scope',
})

# Typage
df_social['year']                 = df_social['year'].astype(int)
df_social['social_media_min_day'] = df_social['social_media_min_day'].astype(float)

# Filtrage période d'analyse
df_social = df_social[
    (df_social['year'] >= YEAR_START) & (df_social['year'] <= YEAR_END)
].reset_index(drop=True)

# Vérification complétude de la série temporelle
years_present  = sorted(df_social['year'].unique())
years_missing  = [y for y in range(YEAR_START, YEAR_END+1) if y not in years_present]

print(f'\n✅ Shape : {df_social.shape}')
print(f'   Années : {years_present}')
print(f'   Manquantes : {years_missing if years_missing else "aucune ✅"}')
print()
print(df_social[['year','social_media_min_day','social_media_hrs_day']].to_string())

DATASET 1 — Temps réseaux sociaux (GWI mondial)
   📂 gwi_gwi_social_media_global_trend_2026-03-15.csv  →  (13, 7)

✅ Shape : (13, 7)
   Années : [np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
   Manquantes : aucune ✅

    year  social_media_min_day  social_media_hrs_day
0   2012                  90.0                  1.50
1   2013                 100.0                  1.67
2   2014                 111.0                  1.85
3   2015                 116.0                  1.93
4   2016                 122.0                  2.03
5   2017                 135.0                  2.25
6   2018                 142.0                  2.37
7   2019                 144.0                  2.40
8   2020                 145.0                  2.42
9   2021                 147.0                  2.45
10  2022                 147.0       

In [18]:
save_processed(df_social, 'social_media_time_global')

   💾 social_media_time_global.csv  →  (13, 7)


'../02_data/processed/social_media_time_global.csv'

---
## 📊 3. Dataset 2 — Temps médias numériques USA (eMarketer 2008-2018)

**FR :** Feature engineering : on somme mobile + desktop + autres pour créer `digital_total_min`.
Toutes les durées sont converties en minutes (harmonisation).

**EN :** Feature engineering: sum 3 device columns → `digital_total_min`. Convert hrs → min.

In [19]:
print('=' * 55)
print('DATASET 2 — Temps médias numériques USA (eMarketer)')
print('=' * 55)

df_digital_raw = load_latest_raw('owid_digital_media_time_usa')
df_digital = df_digital_raw.copy()

# Renommage — les noms OWID bruts encodent la source dans le nom de colonne
df_digital = df_digital.rename(columns={
    'entity':                                               'country',
    'code':                                                 'country_code',
    'mobile__bond_internet_trends__2019':                   'digital_mobile_hrs',
    'desktop_laptop__bond_internet_trends__2019':           'digital_desktop_hrs',
    'other_connected_devices__bond_internet_trends__2019':  'digital_other_hrs',
})

# Feature engineering : total des 3 supports
df_digital['digital_total_hrs'] = (
    df_digital['digital_mobile_hrs'] +
    df_digital['digital_desktop_hrs'] +
    df_digital['digital_other_hrs']
)

# Harmonisation heures → minutes (règle du projet : tout en minutes)
df_digital['digital_total_min']   = (df_digital['digital_total_hrs']   * 60).round(1)
df_digital['digital_mobile_min']  = (df_digital['digital_mobile_hrs']  * 60).round(1)
df_digital['digital_desktop_min'] = (df_digital['digital_desktop_hrs'] * 60).round(1)

df_digital['year'] = df_digital['year'].astype(int)
df_digital = df_digital.sort_values('year').reset_index(drop=True)

print(f'\n✅ Shape : {df_digital.shape}')
print(f'   Couverture : {df_digital["year"].min()} → {df_digital["year"].max()}')
print(f'   ⚠️  Série limitée à 2008-2018 (source eMarketer via OWID)')
print()
print(df_digital[['year','digital_mobile_hrs','digital_desktop_hrs',
                   'digital_other_hrs','digital_total_hrs','digital_total_min']].to_string())

DATASET 2 — Temps médias numériques USA (eMarketer)
   📂 owid_digital_media_time_usa_2026-03-15.csv  →  (11, 6)

✅ Shape : (11, 10)
   Couverture : 2008 → 2018
   ⚠️  Série limitée à 2008-2018 (source eMarketer via OWID)

    year  digital_mobile_hrs  digital_desktop_hrs  digital_other_hrs  digital_total_hrs  digital_total_min
0   2008                 0.3                  2.2                0.2                2.7              162.0
1   2009                 0.3                  2.3                0.3                2.9              174.0
2   2010                 0.4                  2.4                0.4                3.2              192.0
3   2011                 0.8                  2.6                0.3                3.7              222.0
4   2012                 1.6                  2.5                0.3                4.4              264.0
5   2013                 2.3                  2.3                0.3                4.9              294.0
6   2014                 2.6 

In [20]:
save_processed(df_digital, 'digital_media_time_usa')

   💾 digital_media_time_usa.csv  →  (11, 10)


'../02_data/processed/digital_media_time_usa.csv'

---
## 📊 4. Dataset 3 — MAU plateformes (Facebook, TikTok, YouTube...)

**FR :** On produit deux formats du même dataset :
- **Long** : une ligne par (plateforme × année) → standard pour seaborn
- **Wide** : une colonne par plateforme → pour corrélations et Power BI

**EN :** Two formats: long (seaborn-ready) and wide (correlations/Power BI).

In [21]:
print('=' * 55)
print('DATASET 3 — MAU plateformes (Facebook, TikTok...)')
print('=' * 55)

df_platforms_raw = load_latest_raw('gwi_platforms_mau_historical')
df_platforms = df_platforms_raw.copy()

# Typage et filtrage
df_platforms['year']         = df_platforms['year'].astype(int)
df_platforms['mau_millions'] = df_platforms['mau_millions'].astype(float)
df_platforms = df_platforms[
    (df_platforms['year'] >= YEAR_START) & (df_platforms['year'] <= YEAR_END)
].sort_values(['platform','year']).reset_index(drop=True)

# Pivot long → wide (une colonne par plateforme)
# NaN = plateforme inexistante cette année-là (ex: TikTok avant 2018)
df_platforms_wide = df_platforms.pivot_table(
    index='year',
    columns='platform',
    values='mau_millions'
).reset_index()
df_platforms_wide.columns.name = None

print(f'\n✅ Format long  : {df_platforms.shape}')
print(f'   Format wide  : {df_platforms_wide.shape}')
print(f'   Plateformes  : {sorted(df_platforms["platform"].unique())}')
print()
print(df_platforms_wide.to_string())

DATASET 3 — MAU plateformes (Facebook, TikTok...)
   📂 gwi_platforms_mau_historical_2026-03-15.csv  →  (74, 5)

✅ Format long  : (70, 5)
   Format wide  : (13, 7)
   Plateformes  : ['Facebook', 'Instagram', 'Snapchat', 'TikTok', 'X (Twitter)', 'YouTube']

    year  Facebook  Instagram  Snapchat  TikTok  X (Twitter)  YouTube
0   2012    1056.0       30.0       NaN     NaN        185.0    800.0
1   2013    1230.0       90.0       NaN     NaN        241.0   1000.0
2   2014    1393.0      200.0     100.0     NaN        288.0   1000.0
3   2015    1591.0      400.0     200.0     NaN        320.0   1500.0
4   2016    1860.0      600.0     301.0     NaN        319.0   1500.0
5   2017    2129.0      800.0     187.0     NaN        330.0   1500.0
6   2018    2320.0     1000.0     190.0   271.0        336.0   1900.0
7   2019    2498.0     1082.0     218.0   508.0        339.0   2000.0
8   2020    2797.0     1221.0     265.0   689.0        353.0   2291.0
9   2021    2912.0     1393.0     319.0  100

In [22]:
save_processed(df_platforms,      'platforms_mau_long')
save_processed(df_platforms_wide, 'platforms_mau_wide')

   💾 platforms_mau_long.csv  →  (70, 5)
   💾 platforms_mau_wide.csv  →  (13, 7)


'../02_data/processed/platforms_mau_wide.csv'

---
## 📊 5. Dataset 4 — Comparaison internationale 2024 (GWI par pays)

**FR :** Snapshot du temps passé sur les réseaux sociaux par pays en 2024.
On ajoute une variable régionale pour colorier les visualisations.

**EN :** 2024 international snapshot. Adding region for chart coloring.

In [23]:
print('=' * 55)
print('DATASET 4 — Comparaison internationale 2024 (GWI)')
print('=' * 55)

df_countries_raw = load_latest_raw('gwi_gwi_social_media_by_country_2024')
df_countries = df_countries_raw.copy()

df_countries = df_countries.rename(columns={
    'entity':               'country',
    'social_media_minutes': 'social_media_min_day',
    'social_media_hours':   'social_media_hrs_day',
})

# Feature engineering : variable régionale pour les graphiques
region_map = {
    'Brazil':'Amérique Latine', 'Mexico':'Amérique Latine',
    'United States':'Amérique du Nord',
    'France':'Europe', 'United Kingdom':'Europe', 'Germany':'Europe',
    'Australia':'Océanie',
    'Japan':'Asie', 'South Korea':'Asie', 'India':'Asie', 'Philippines':'Asie',
    'South Africa':'Afrique', 'Kenya':'Afrique',
}
df_countries['region'] = df_countries['country'].map(region_map)
df_countries = df_countries.sort_values('social_media_min_day', ascending=False).reset_index(drop=True)

print(f'\n✅ Shape : {df_countries.shape}')
print()
print(df_countries[['country','region','social_media_min_day','social_media_hrs_day']].to_string())

DATASET 4 — Comparaison internationale 2024 (GWI)
   📂 gwi_gwi_social_media_by_country_2024_2026-03-15.csv  →  (13, 6)

✅ Shape : (13, 7)

           country            region  social_media_min_day  social_media_hrs_day
0           Brazil   Amérique Latine                   229                  3.82
1            Kenya           Afrique                   223                  3.72
2     South Africa           Afrique                   221                  3.68
3      Philippines              Asie                   214                  3.57
4           Mexico   Amérique Latine                   185                  3.08
5            India              Asie                   182                  3.03
6    United States  Amérique du Nord                   129                  2.15
7        Australia           Océanie                   111                  1.85
8           France            Europe                   105                  1.75
9   United Kingdom            Europe               

In [24]:
save_processed(df_countries, 'social_media_by_country_2024')

   💾 social_media_by_country_2024.csv  →  (13, 7)


'../02_data/processed/social_media_by_country_2024.csv'

---
## 📊 6. Dataset 5 — TV France (CNC/Médiamétrie 2009-2024)

**FR :** Ce dataset a déjà été produit et nettoyé directement par le script
`collect_cnc_mediametrie.py` (Module 1). Il n'y a **rien à nettoyer ici**.
On charge simplement les fichiers et on vérifie leur contenu.

**⚠️ Pré-requis :** avoir lancé ce script avant d'ouvrir ce notebook :
```bash
python 01_data_collection/collect_cnc_mediametrie.py
```

**EN :** Already cleaned by `collect_cnc_mediametrie.py`. Just load and verify.

In [25]:
print('=' * 55)
print('DATASET 5 — TV France (CNC/Médiamétrie 2009-2024)')
print('=' * 55)

tv_wide_path = os.path.join(PROCESSED_DIR, 'tv_audiences_france.csv')
tv_long_path = os.path.join(PROCESSED_DIR, 'tv_audiences_france_long.csv')

if not os.path.exists(tv_wide_path):
    print('❌ Fichier manquant !')
    print('   Lance : python 01_data_collection/collect_cnc_mediametrie.py')
else:
    # Chargement direct — déjà nettoyé par collect_cnc_mediametrie.py
    df_tv      = pd.read_csv(tv_wide_path)
    df_tv_long = pd.read_csv(tv_long_path)

    print(f'\n✅ Format wide  : {df_tv.shape}')
    print(f'   Format long  : {df_tv_long.shape}')
    print(f'   Années       : {df_tv["year"].min()} → {df_tv["year"].max()}')
    print()
    print(df_tv[['year','tv_min_4_14','tv_min_15_49','tv_min_50plus',
                 'gap_50plus_vs_1549','methodological_break']].to_string())

    # Vérification rapide H3
    gap_2012 = df_tv.loc[df_tv['year']==2012, 'gap_50plus_vs_1549'].values
    gap_2023 = df_tv.loc[df_tv['year']==2023, 'gap_50plus_vs_1549'].values
    if len(gap_2012) and len(gap_2023):
        print(f'\n📐 Vérification H3 (écart générationnel TV) :')
        print(f'   2012 : {gap_2012[0]:.0f} min  |  2023 : {gap_2023[0]:.0f} min')
        print(f'   → H3 {"CONFIRMÉE ✅" if gap_2023[0] > gap_2012[0] else "INFIRMÉE ❌"}')

DATASET 5 — TV France (CNC/Médiamétrie 2009-2024)

✅ Format wide  : (16, 11)
   Format long  : (48, 8)
   Années       : 2009 → 2024

    year  tv_min_4_14  tv_min_15_49  tv_min_50plus  gap_50plus_vs_1549                                                          methodological_break
0   2009        131.0         179.0          266.0                87.0                                                                           NaN
1   2010        132.0         185.0          274.0                89.0                                                                           NaN
2   2011        138.0         196.0          299.0               103.0                                   ¹ TV différé +7j (enregistrement personnel)
3   2012        135.0         199.0          302.0               103.0                                                                           NaN
4   2013        129.0         191.0          302.0               111.0                                                   

---
## 🔗 7. Dataset consolidé — Jointure GWI × TV France (base pour H1)

**FR :** On fusionne le temps réseaux sociaux (GWI) avec les audiences TV France (CNC)
sur la période commune. On ajoute les **indices base 100 = 2012** pour visualiser
les évolutions relatives des deux séries sur le même graphique.

Formule indice base 100 : `(valeur_t / valeur_2012) × 100`
→ Si l'indice TV = 58 en 2023, cela signifie que le temps TV a baissé de 42% depuis 2012.

**EN :** Inner join on year. Base-100 indices (2012=100) for relative comparison.

In [26]:
print('=' * 55)
print('DATASET CONSOLIDÉ — GWI × TV France (pour H1)')
print('=' * 55)

# Inner join sur l'année = intersection des deux séries
# → donne la période commune 2012-2024
df_h1 = pd.merge(
    df_social[['year','social_media_min_day']],
    df_tv[['year','tv_min_15_49','tv_min_50plus','gap_50plus_vs_1549','methodological_break']],
    on='year',
    how='inner'
)

# Indices base 100 = 2012
# Permettent de comparer deux séries avec des unités / ordres de grandeur différents
base_year = 2012
if base_year in df_h1['year'].values:
    base_social = df_h1.loc[df_h1['year']==base_year, 'social_media_min_day'].values[0]
    base_tv     = df_h1.loc[df_h1['year']==base_year, 'tv_min_15_49'].values[0]
    df_h1['idx_social_100'] = (df_h1['social_media_min_day'] / base_social * 100).round(1)
    df_h1['idx_tv_100']     = (df_h1['tv_min_15_49']         / base_tv     * 100).round(1)

print(f'\n✅ Dataset H1 : {df_h1.shape}')
print(f'   Période commune : {df_h1["year"].min()} → {df_h1["year"].max()}')
print()
print(df_h1.to_string())

DATASET CONSOLIDÉ — GWI × TV France (pour H1)

✅ Dataset H1 : (13, 8)
   Période commune : 2012 → 2024

    year  social_media_min_day  tv_min_15_49  tv_min_50plus  gap_50plus_vs_1549                                                          methodological_break  idx_social_100  idx_tv_100
0   2012                  90.0         199.0          302.0               103.0                                                                           NaN           100.0       100.0
1   2013                 100.0         191.0          302.0               111.0                                                                           NaN           111.1        96.0
2   2014                 111.0         183.0          302.0               119.0                                        ² Rattrapage TV sur téléviseur intégré           123.3        92.0
3   2015                 116.0         182.0          307.0               125.0                                                                         

In [27]:
save_processed(df_h1, 'consolidated_h1_social_vs_tv')

   💾 consolidated_h1_social_vs_tv.csv  →  (13, 8)


'../02_data/processed/consolidated_h1_social_vs_tv.csv'

---
## 📋 8. Rapport final

**FR :** Récapitulatif de tous les fichiers dans `02_data/processed/`.
Vérifie que les 8 fichiers attendus sont bien présents avant de passer à l'EDA.

**EN :** Final summary. Verify all 8 expected files exist before moving to EDA.

---
## 📊 7. Dataset 5b — TV France 15-34 ans & dataset consolidé

**FR :**
La tranche 15-49 ans du CNC est trop large pour notre sujet.
On charge ici les deux datasets supplémentaires produits par `collect_cnc_mediametrie.py` v2.1 :
- `tv_audiences_france_1534.csv` — série 15-34 ans (2013-2024, sources multiples)
- `tv_audiences_france_consolidated.csv` — jointure 15-34 / 15-49 / 50+ (2013-2024)

**2024 = NaN documenté** : rupture méthodologique Médiamétrie (nouvelle méthode complète,
non comparable à la série historique). La colonne `note_2024` contient l'explication.

**EN :** Load fine-grained 15-34 series + consolidated dataset. 2024 = documented NaN.

In [29]:
# ── Chargement des datasets TV 15-34 ans ──────────────────────────────────────
# Ces fichiers sont produits par collect_cnc_mediametrie.py v2.1
# Ils doivent exister dans processed/ avant de lancer ce notebook

path_1534 = os.path.join(PROCESSED_DIR, 'tv_audiences_france_1534.csv')
path_cons  = os.path.join(PROCESSED_DIR, 'tv_audiences_france_consolidated.csv')

df_tv_1534 = pd.read_csv(path_1534)
df_tv_cons = pd.read_csv(path_cons)

print('DATASET 5b — TV France 15-34 ans')
print('=' * 55)
print(f'  tv_audiences_france_1534.csv      : {df_tv_1534.shape}')
print(f'  tv_audiences_france_consolidated  : {df_tv_cons.shape}')
print()

# ── Vérification qualité ──────────────────────────────────────────────────────
print('Valeurs manquantes par colonne (1534) :')
print(df_tv_1534.isnull().sum()[df_tv_1534.isnull().sum() > 0].to_string())
print()

# ── Le NaN 2024 est documenté — on vérifie qu'il porte bien sa note ──────────
row_2024 = df_tv_1534[df_tv_1534['year']==2024]
if not row_2024.empty:
    print('✅ 2024 présent comme NaN documenté')
    print(f'   note_2024 : {str(row_2024["note_2024"].values[0])[:80]}...')
    print(f'   confidence : {row_2024["confidence"].values[0]}')
print()

# ── Aperçu du dataset consolidé ──────────────────────────────────────────────
print('Dataset consolidé (15-34 / 15-49 / 50+) :')
cols_show = ['year','tv_min_15_34','tv_min_15_49','tv_min_50plus',
             'gap_50plus_vs_1534','gap_50plus_vs_1549','is_break_year']
print(df_tv_cons[cols_show].to_string(index=False))
print()

# ── Stats descriptives sur la série 15-34 (hors NaN) ─────────────────────────
df_1534_clean = df_tv_1534.dropna(subset=['tv_min_15_34'])
print('Stats descriptives — TV 15-34 ans (2013-2023) :')
print(df_1534_clean[['tv_min_15_34']].describe().round(1).to_string())
tv_start = df_1534_clean.iloc[0]['tv_min_15_34']
tv_end   = df_1534_clean.iloc[-1]['tv_min_15_34']
print(f'  Chute 2013→2023 : {tv_start:.0f} → {tv_end:.0f} min  (−{tv_start-tv_end:.0f} min, −{(tv_start-tv_end)/tv_start*100:.0f}%)')
print()
print('✅ Datasets 5b chargés et validés')

FileNotFoundError: [Errno 2] No such file or directory: '../02_data/processed/tv_audiences_france_1534.csv'

In [ ]:
print('=' * 65)
print('📋 RAPPORT DE NETTOYAGE / CLEANING REPORT')
print('=' * 65)

# Liste et caractérise tous les fichiers produits
processed_files = sorted(glob.glob(os.path.join(PROCESSED_DIR, '*.csv')))
print(f'\n{len(processed_files)} fichiers dans processed/\n')
print(f'{"Fichier":<45} {"Lignes":<8} {"Cols":<6} {"NaN"}')
print('─' * 68)
for fp in processed_files:
    df_check  = pd.read_csv(fp)
    nan_count = df_check.isnull().sum().sum()
    print(f'{os.path.basename(fp):<45} {df_check.shape[0]:<8} {df_check.shape[1]:<6} {nan_count}')
print('─' * 68)

# Vérification des 8 fichiers attendus
print('\nVérification des fichiers attendus :')
expected = [
    'social_media_time_global.csv',
    'digital_media_time_usa.csv',
    'platforms_mau_long.csv',
    'platforms_mau_wide.csv',
    'social_media_by_country_2024.csv',
    'tv_audiences_france.csv',
    'tv_audiences_france_long.csv',
    'consolidated_h1_social_vs_tv.csv',
]
existing = [os.path.basename(f) for f in processed_files]
for fname in expected:
    icon = '✅' if fname in existing else '❌ MANQUANT'
    print(f'   {icon}  {fname}')

all_ok = all(f in existing for f in expected)
print()
if all_ok:
    print('🎉 Tout est prêt ! → Prochaine étape : 02_eda_exploratory.ipynb')
else:
    print('⚠️  Certains fichiers manquent. Vérifie les étapes ci-dessus.')